# E003 — Audio: GMM-UBM + MAP adaptation

Same features as E002 (MFCC 13+Δ+ΔΔ, CMN). The key difference is how we
build the target model:

- **E002:** independent GMM trained only on target frames (few data → fragile)
- **E003:** UBM trained on all non-target frames, then MAP-adapted to target

MAP adaptation formula (means only, standard approach):
```
α_k  = n_k / (n_k + r)                              # adaptation weight
μ_k' = α_k · μ̂_k_target + (1 − α_k) · μ_k_UBM    # adapted mean
```
where n_k = expected count of target frames assigned to component k,
and r = relevance factor (controls prior strength, typically 16).

In [ ]:
from pathlib import Path
import sys
import copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve, auc
from scipy.special import logsumexp

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

In [ ]:
def extract_mfcc(wav_path: Path, n_mfcc: int = 13) -> np.ndarray:
    y, sr  = librosa.load(wav_path, sr=None, mono=True)
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    mfcc   = np.vstack([mfcc, delta, delta2]).T
    mfcc  -= mfcc.mean(axis=0)  # CMN
    return mfcc


def find_wav(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def extract_batch(df, data_dir: Path):
    all_mfcc, all_labels = [], []
    for _, row in df.iterrows():
        mfcc = extract_mfcc(find_wav(row["stem"], data_dir))
        all_mfcc.append(mfcc)
        all_labels.extend([row["label"]] * len(mfcc))
    return np.vstack(all_mfcc), np.array(all_labels)

In [ ]:
def train_ubm(X: np.ndarray, n_components: int = 32, seed: int = 67) -> GaussianMixture:
    """Train UBM on non-target frames."""
    gmm = GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    )
    gmm.fit(X)
    return gmm


def map_adapt(ubm: GaussianMixture, X_target: np.ndarray, r: float = 16.0) -> GaussianMixture:
    """
    MAP-adapt UBM means toward target data. Weights and covariances stay fixed.

    Steps:
    1. E-step: compute posterior P(k | frame) for each target frame
    2. Accumulate sufficient statistics: n_k, μ̂_k
    3. Compute adaptation coefficient α_k = n_k / (n_k + r)
    4. New means: α_k * μ̂_k + (1-α_k) * μ_k_UBM
    """
    # E-step: log responsibilities (T, K)
    log_prob = ubm._estimate_log_prob(X_target)          # (T, K) log p(x|k)
    log_weights = np.log(ubm.weights_)                   # (K,)
    log_resp = log_prob + log_weights                     # (T, K)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)                              # (T, K)  posterior

    # Sufficient statistics
    n_k = resp.sum(axis=0)                               # (K,) expected counts
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)  # (K, D) weighted means

    # MAP update
    alpha = n_k / (n_k + r)                              # (K,)
    adapted_means = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_

    adapted = copy.deepcopy(ubm)
    adapted.means_ = adapted_means
    return adapted


def score_utterance(wav_path: Path, adapted: GaussianMixture, ubm: GaussianMixture) -> float:
    """LLR score: log P(x|adapted) - log P(x|UBM), averaged over frames."""
    mfcc = extract_mfcc(wav_path)
    llr = adapted.score_samples(mfcc) - ubm.score_samples(mfcc)
    return float(llr.mean())

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0
SEED = 67

oof_scores = np.full(len(manifest), np.nan)
fold_results = []

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    train_df = manifest.loc[train_idx]
    val_df   = manifest.loc[val_idx]
    print(f"Fold {fold_id}: train={len(train_df)}, val={len(val_df)}")

    print("  Extracting features...")
    X_train, y_train = extract_batch(train_df, DATA)
    X_nt = X_train[y_train == 0]
    X_t  = X_train[y_train == 1]
    print(f"  Frames: {len(X_t)} target, {len(X_nt)} non-target")

    print("  Training UBM...")
    ubm = train_ubm(X_nt, n_components=UBM_COMPONENTS, seed=SEED)

    print("  MAP adapting to target...")
    adapted = map_adapt(ubm, X_t, r=MAP_R)

    print("  Scoring val fold...")
    for idx, row in val_df.iterrows():
        wav = find_wav(row["stem"], DATA)
        oof_scores[idx] = score_utterance(wav, adapted, ubm)

    val_scores = oof_scores[val_idx]
    val_labels = manifest.loc[val_idx, "label"].to_numpy()
    eer, _     = compute_eer(val_scores[val_labels==1], val_scores[val_labels==0])
    min_dcf, _ = compute_min_dcf(val_scores[val_labels==1], val_scores[val_labels==0])
    fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
    print(f"  → EER = {eer*100:.2f}%, min-DCF = {min_dcf:.4f}\n")

print("Done.")

In [ ]:
import pandas as pd

results_df = pd.DataFrame(fold_results)
mean_eer   = results_df["eer"].mean()
std_eer    = results_df["eer"].std()
mean_dcf   = results_df["min_dcf"].mean()
std_dcf    = results_df["min_dcf"].std()

print("=" * 45)
print(f"{'Fold':<8} {'EER [%]':>10} {'min-DCF':>10}")
print("-" * 45)
for _, r in results_df.iterrows():
    print(f"{int(r.fold):<8} {r.eer*100:>10.2f} {r.min_dcf:>10.4f}")
print("-" * 45)
print(f"{'mean±std':<8} {mean_eer*100:>7.2f}±{std_eer*100:.2f} {mean_dcf:>7.4f}±{std_dcf:.4f}")
print("=" * 45)

y_all = manifest["label"].to_numpy()
eer_oof, _   = compute_eer(oof_scores[y_all==1], oof_scores[y_all==0])
dcf_oof, thr = compute_min_dcf(oof_scores[y_all==1], oof_scores[y_all==0])
print(f"\nOOF overall: EER = {eer_oof*100:.2f}%, min-DCF = {dcf_oof:.4f}, threshold = {thr:.3f}")

print(f"\nE001 (static MFCC):  EER = 17.92 ± 7.81%")
print(f"E002 (MFCC+Δ+ΔΔ):   EER = 10.09 ± 1.81%")
print(f"E003 (UBM+MAP):      EER = {mean_eer*100:.2f} ± {std_eer*100:.2f}%")
print(f"Change vs E002: {(mean_eer*100 - 10.09):+.2f}% EER")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
bins = np.linspace(oof_scores.min(), oof_scores.max(), 40)
ax.hist(oof_scores[y_all==0], bins=bins, alpha=0.6, color="steelblue", label="non-target", density=True)
ax.hist(oof_scores[y_all==1], bins=bins, alpha=0.7, color="tomato",    label="target",     density=True)
ax.axvline(thr, color="green", ls="--", lw=2, label=f"threshold = {thr:.2f}")
ax.set_xlabel("OOF score (LLR)")
ax.set_title("Score distributions")
ax.legend()

ax = axes[1]
fpr, tpr, _ = roc_curve(y_all, oof_scores)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color="tomato", lw=2, label=f"AUC = {roc_auc:.3f}")
ax.plot([0,1],[0,1],"k--",lw=1)
ax.set_xlabel("FAR")
ax.set_ylabel("1 - FRR")
ax.set_title("ROC curve (OOF)")
ax.legend()

plt.suptitle(f"E003 — GMM-UBM+MAP  |  EER = {eer_oof*100:.2f}%, min-DCF = {dcf_oof:.4f}")
plt.tight_layout()
plt.show()